# Installing the necessary packages

In [ ]:
! pip install arabert
! pip install transformers
! pip install farasapy
! pip install pyarabic

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Arabic_Dialect_Classification/cleaned_dataset.csv", engine="python")

In [ ]:
data

In [ ]:
data

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Count the occurrences of each class in your dataset
label_counts = data['SpeakerDialect'].value_counts()  # Replace 'label' with your actual column name

# Plot the distribution
sns.barplot(x=label_counts.index, y=label_counts.values)
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Class Distribution")
plt.xticks(rotation=45)  # Rotate labels if necessary
plt.show()


In [ ]:
data

In [ ]:
data["SpeakerDialect"].unique()


In [ ]:
# Mapping dialects to numbers
map_label = {
    'Najdi': 0,
    'More than 1 speaker اكثر من متحدث': 1,
    'Unknown': 2,
    'Khaliji': 3,
    'Hijazi': 4,
    'ModernStandardArabic': 5,
    'Notapplicable': 6,
    'Egyptian': 7,
    'Levantine': 8,
    'Yemeni': 9,
    'Maghrebi': 10,
    'Janubi': 11,
    'Shamali': 12,
    'Iraqi': 13
}

# Mapping numbers back to dialects
label_map = {v: k for k, v in map_label.items()}
data['SpeakerDialect'] = data['SpeakerDialect'].map(map_label)

print(map_label)
print(label_map)


In [ ]:
data

In [ ]:
from transformers import AutoTokenizer
model_name="UBC-NLP/MARBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
from sklearn.model_selection import train_test_split
# Rename columns if necessary
data = data.rename(columns={'SpeakerDialect': 'dialect', 'Cleaned_text': 'text'})


In [ ]:
data

In [ ]:
data

In [ ]:
# Split into train (80%) and test (20%)
dataset, test_data = train_test_split(data, test_size=0.2, random_state=42, stratify=data['dialect'])

# Display dataset shapes
print("Train set size:", dataset.shape)
print("Test set size:", test_data.shape)

In [ ]:
dataset

In [ ]:
test_data=test_data[test_data['dialect'].isnull()==False]
dataset=dataset[dataset['dialect'].isnull()==False]

In [ ]:
dataset = dataset[dataset["text"].notna() & (dataset["text"].str.strip() != "")]
test_data = test_data[test_data["text"].notna() & (test_data["text"].str.strip() != "")]


In [ ]:
dataset

In [ ]:
print(dataset['dialect'].value_counts())


In [ ]:
# Initialize AraBERT Preprocessor
from arabert.preprocess import ArabertPreprocessor
model_name1 = "aubmindlab/bert-base-arabert"
arabert_prep = ArabertPreprocessor(model_name=model_name1)


In [ ]:
dataset["text"]=dataset["text"].apply(lambda x:arabert_prep.preprocess(x))

In [ ]:
test_data["text"]=test_data["text"].apply(lambda x:arabert_prep.preprocess(x))

In [ ]:
dataset

In [ ]:
test_data


# Importing the necessary packages

In [ ]:
from arabert.preprocess import ArabertPreprocessor
from sklearn.metrics import (accuracy_score, f1_score,recall_score)
from torch.utils.data import  Dataset
from transformers import (AutoConfig, AutoModelForSequenceClassification,
                        AutoTokenizer, BertTokenizer, Trainer,
                        TrainingArguments)
from transformers.data.processors.utils import InputFeatures

In [ ]:
#chose bert model
model_name = 'UBC-NLP/MARBERT'
#asafaya/bert-base-arabic
#UBC-NLP/ARBERT
#UBC-NLP/MARBERT
#bert-base-multilingual-uncased
num_labels = 14
max_length = 120

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


## To work using PyTorch we need to create a classification dataset to load the data

In [ ]:
# ✅ Define classification dataset
class ClassificationDataset(Dataset):
    def __init__(self, text, target, tokenizer, max_len):
        self.text = text
        self.target = target
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, item):
        text = str(self.text[item])
        text = " ".join(text.split())  # Normalize spaces

        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.target[item], dtype=torch.long),
        }


In [ ]:
#dataset['dialect'] = dataset['dialect'].map(map_label)
#test_data['dialect'] = test_data['dialect'].map(map_label)


In [ ]:
#test_data=test_data[test_data['dialect'].isnull()==False]
#dataset=dataset[dataset['dialect'].isnull()==False]

In [ ]:
dataset['dialect'] = dataset['dialect'].astype(int)
test_data['dialect'] = test_data['dialect'].astype(int)

In [ ]:
dataset.info()

In [ ]:
dataset

## Creating datasets

In [ ]:
# ✅ Create datasets
train_dataset = ClassificationDataset(dataset["text"].tolist(), dataset["dialect"].tolist(), tokenizer, max_length)
test_dataset = ClassificationDataset(test_data["text"].tolist(), test_data["dialect"].tolist(), tokenizer, max_length)

print(f"✅ Train dataset size: {len(train_dataset)}")
print(f"✅ Test dataset size: {len(test_dataset)}")

In [ ]:
(len(train_dataset))

## Create a function that return a pretrained model ready to do classification

In [ ]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_name, return_dict=True, num_labels=num_labels)

## Metrics

In [ ]:
def compute_metrics(p): #p should be of type EvalPrediction
  preds = np.argmax(p.predictions, axis=1)
  assert len(preds) == len(p.label_ids)
  macro_f1 = f1_score(p.label_ids,preds,average='macro')
  macro_recall = recall_score(p.label_ids,preds,average='macro')
  acc = accuracy_score(p.label_ids,preds)
  return {
      'macro_f1' : macro_f1,
      'accuracy': acc,
      'recall':macro_recall
  }

## Training arguments

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback
import torch
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Arabic_Dialect_Classification/MARBERT",
    adam_epsilon=1e-8,
    learning_rate=5e-6,
    fp16=True,  # استخدام حساب النقطة العائمة 16 بت إذا كان لديك V100/T4
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    warmup_ratio=0.1,  # استخدام Warmup لتحسين الاستقرار
    do_eval=True,
    weight_decay=0.01 ,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=5,  # تقليل عدد النماذج المحفوظة لتوفير المساحة
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to=[],
)

## Creating the trainer

In [ ]:
trainer = Trainer(
    model = model_init(),
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Tarining

In [ ]:
trainer.train()

## Saving the model

In [ ]:
import os
model_save_path = '/content/drive/MyDrive/Arabic_Dialect_Classification/model_checkpoint_marbert'
# Set the model configuration for label mappings
trainer.model.config.label2id = map_label
trainer.model.config.id2label = label_map
trainer.save_model(os.path.join(model_save_path, 'checkpoint-2'))
train_dataset.tokenizer.save_pretrained(os.path.join(model_save_path, 'checkpoint-2'))